In [1]:
import pandas as pd
import numpy as np

# 1. LOAD DATA
# Ensure these two files are in the same directory as your notebook
df = pd.read_csv('C:/Users/vinay/OneDrive - Indian Institute of Technology Bhubaneswar/Desktop/sedrica assignment/city/crossing_evidence.csv')
ref_df = pd.read_csv('C:/Users/vinay/OneDrive - Indian Institute of Technology Bhubaneswar/Desktop/sedrica assignment/city/crossing_reference.csv')

# Merge for evaluation and compute message age
df = pd.merge(df, ref_df, on=['episode', 'frame'])
df['original_message_age'] = round(df['time_s'] - df['v2x_sample_time_s'], 2)

# 2. DEFINE RULES & THRESHOLDS
RED_THRESH = 0.5
PERSON_THRESH = 0.5
STALE_THRESH = 0.5 

def get_baseline_decision(row):
    if row['person_score'] > PERSON_THRESH: return 'STOP'
    v_red = row['red_score'] > RED_THRESH
    x_red = row['v2x_light'] == 'red'
    if v_red and x_red: return 'STOP'
    elif v_red != x_red: return 'SLOW'
    else: return 'GO'

def get_revised_decision(row):
    if row['person_score'] > PERSON_THRESH: return 'STOP'
    v_red = row['red_score'] > RED_THRESH
    x_red = row['v2x_light'] == 'red'
    is_stale = row['msg_age'] > STALE_THRESH
    
    if v_red:
        if x_red: return 'STOP'
        else: return 'STOP' if is_stale else 'SLOW'
    else:
        if x_red: return 'SLOW' if is_stale else 'SLOW'
        else: return 'SLOW' if is_stale else 'GO'

# Apply rules to unperturbed data
df['msg_age'] = df['original_message_age']
df['baseline_decision'] = df.apply(get_baseline_decision, axis=1)
df['revised_decision'] = df.apply(get_revised_decision, axis=1)

# 3. STOPPING DISTANCE CALCULATION
df['d_stop'] = df['speed_mps'] * 0.15 + (df['speed_mps']**2) / (2 * 0.8)
print("=== TIMELINESS CHECK (UNPERTURBED DATA) ===")
for ep in df['episode'].unique():
    ep_df = df[df['episode'] == ep]
    stop_frames = ep_df[ep_df['revised_decision'] == 'STOP']
    if not stop_frames.empty:
        first_stop = stop_frames.iloc[0]
        print(f"{ep}: STOP @ frame {first_stop['frame']}. Distance to line: {first_stop['distance_to_line_m']}m | Required: {first_stop['d_stop']}m. Safe? {first_stop['distance_to_line_m'] >= first_stop['d_stop']}")
print("\n")

# 4. UNCERTAINTY TEST (20 TRIALS)
results = []
np.random.seed(42) # For reproducible results

for trial in range(20):
    df_trial = df.copy()
    # Inject vision noise
    df_trial['red_score'] = np.clip(df_trial['red_score'] + np.random.normal(0, 0.15, len(df_trial)), 0, 1)
    df_trial['person_score'] = np.clip(df_trial['person_score'] + np.random.normal(0, 0.15, len(df_trial)), 0, 1)
    
    # Simulate V2X drop (15% chance to drop, making data stale)
    v2x_drops = np.random.rand(len(df_trial)) < 0.15
    df_trial['msg_age'] = df_trial['original_message_age']
    df_trial.loc[v2x_drops, 'msg_age'] += 0.6 
    
    df_trial['baseline'] = df_trial.apply(get_baseline_decision, axis=1)
    df_trial['revised'] = df_trial.apply(get_revised_decision, axis=1)
    
    for rule in ['baseline', 'revised']:
        missed = len(df_trial[(df_trial['stop_required'] == 1) & (df_trial[rule] != 'STOP')])
        unnecessary = len(df_trial[(df_trial['stop_required'] == 0) & (df_trial[rule] == 'STOP')])
        results.append({'trial': trial, 'rule': rule, 'missed_stops': missed, 'unnecessary_stops': unnecessary})

# 5. PRINT SUMMARY
res_df = pd.DataFrame(results)
summary = res_df.groupby('rule').agg({'missed_stops': 'mean', 'unnecessary_stops': 'mean'}).round(2)
print("=== 20-TRIAL MONTE CARLO SUMMARY ===")
print("Average Missed STOPs (Lower is safer):")
print(f"  Baseline Rule: {summary.loc['baseline', 'missed_stops']}")
print(f"  Revised Rule:  {summary.loc['revised', 'missed_stops']}")

# Save the final unperturbed timeline for submission
df.to_csv('final_q2_timeline.csv', index=False)
print("\nSuccess! Saved full frame-by-frame timeline to 'final_q2_timeline.csv'")

=== TIMELINESS CHECK (UNPERTURBED DATA) ===
clear_green: STOP @ frame 6. Distance to line: 2.27m | Required: 0.52m. Safe? True
late_red: STOP @ frame 1. Distance to line: 2.67m | Required: 0.52m. Safe? True
person_crossing: STOP @ frame 7. Distance to line: 2.19m | Required: 0.52m. Safe? True


=== 20-TRIAL MONTE CARLO SUMMARY ===
Average Missed STOPs (Lower is safer):
  Baseline Rule: 5.45
  Revised Rule:  3.45

Success! Saved full frame-by-frame timeline to 'final_q2_timeline.csv'
